# Norris Phase 3 — resume saved dev baseline evaluation

This is a corrected, restart-safe continuation notebook. It does **not** contain the old Phase 2 generation cells.

1. Upload this notebook to Colab and use **File → Save a copy in Drive**.
2. Select **Runtime → Change runtime type → A100 GPU**, then connect.
3. Run the four code cells below in order. Authorize the Google Drive mount when prompted.
4. Attach the resulting `Norris_Phase3_Dev_Evaluation_Evidence_....zip` in the chat.

No NLLB model is downloaded or rerun. Only the saved **dev** files are opened. A completed evaluation is verified and reused when available. The Python 3.11 evaluation environment is separate from Colab's main environment.

Confirmed in the previous generation log: 997 records per direction per condition; 3,988 current unsplit outputs match the archived translations exactly; sentence splitting changed 96 / 93 / 85 / 103 outputs for En→Tl / Tl→En / En→Es / Es→En. These are generation checks, not measured quality gains.

This notebook finishes the **baseline evaluation checkpoint**. The LLM pilots, ICAI overlap audit, and protocol freeze must still be evidenced before Phase 3 can be reported as complete. GPT API access was previously pending. No external human raters are to be recruited.


In [ ]:
# 1 — Restore and verify the saved dev run. No previous variables needed.
from pathlib import Path
from google.colab import drive
import hashlib
import importlib.util
import json
import os
import subprocess
import sys

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/norris')
RUN_DIR = ROOT / 'outputs/generations/baselines/phase3_dev_f4a3af94ba1e72fd'

EVALUATOR_SOURCE = '"""Resume-safe dev-only baseline evaluation; no NLLB or post-editor generation."""\nimport os\nos.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\n\nimport sys\nimport json\nimport math\nimport hashlib\nfrom pathlib import Path\nfrom importlib.metadata import version, distributions\n\nRUN_NAME = \'phase3_dev_f4a3af94ba1e72fd\'\nEXPECTED_RUN_SHA = \'f4a3af94ba1e72fddc14a8986924fc23e00ae2884dc8d51800ad79e43a6a0563\'\nCOMET_ID = \'Unbabel/wmt22-comet-da\'\nCOMET_REV = \'2760a223ac957f30acfb18c8aa649b01cf1d75f2\'\nDIRECTIONS = [(\'eng_Latn\', \'tgl_Latn\'), (\'tgl_Latn\', \'eng_Latn\'),\n              (\'eng_Latn\', \'spa_Latn\'), (\'spa_Latn\', \'eng_Latn\')]\nEXPECTED_HASHES = {\n \'nllb3.3b_eng_Latn_tgl_Latn_dev_unsplit_control.jsonl\': \'4a17a36f513a5eb90b05d3192179124605bd99c65902bdbdd7bf5463ef091f9f\',\n \'nllb3.3b_eng_Latn_tgl_Latn_dev_sentence_split.jsonl\': \'d6617cb4b12dc23203326970cc5db9d217be6cdfc658e2b4c3eb59f67e252176\',\n \'nllb3.3b_tgl_Latn_eng_Latn_dev_unsplit_control.jsonl\': \'e218625b96cffb539f42e368a26c025b400a457c3f6d35a9625e97de9478abd0\',\n \'nllb3.3b_tgl_Latn_eng_Latn_dev_sentence_split.jsonl\': \'8a0924a378738010fdb13f4a0a5eaa7da8bbda30d3bcc4fa526c26c50c47a3f4\',\n \'nllb3.3b_eng_Latn_spa_Latn_dev_unsplit_control.jsonl\': \'575b6519738da932ae6dc9520f61b6c4ac5db669b81121d304e18b2df81bf43c\',\n \'nllb3.3b_eng_Latn_spa_Latn_dev_sentence_split.jsonl\': \'34b1d8f011d4addeae4dda7049ac58fea35a9beb8d07dd8d0201665f088d0a42\',\n \'nllb3.3b_spa_Latn_eng_Latn_dev_unsplit_control.jsonl\': \'a0eda958cf61561208ebff752bc54b10d41a8b996251b8c02cf1bf7684a840c3\',\n \'nllb3.3b_spa_Latn_eng_Latn_dev_sentence_split.jsonl\': \'291c5e9ca8c0fdbf7108bfeb7949f793939e1cbf66f35238b4165819e95ce088\',\n}\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True,\n                      separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')\n\n\ndef digest(value):\n    return hashlib.sha256(canonical(value)).hexdigest()\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.tmp\')\n    with temporary.open(\'w\', encoding=\'utf-8\') as f:\n        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)\n        f.flush()\n    os.replace(temporary, path)\n\n\ndef read_json(path):\n    with Path(path).open(encoding=\'utf-8\') as f:\n        return json.load(f)\n\n\ndef load_inputs(input_dir):\n    """Validate the saved run against the exact hashes recorded in chat."""\n    input_dir = Path(input_dir)\n    summary = read_json(input_dir / \'generation_summary.json\')\n    spec = read_json(input_dir / \'run_spec.json\')\n    require(summary[\'run_spec_sha256\'] == EXPECTED_RUN_SHA, \'Unexpected generation run.\')\n    require(digest(spec) == EXPECTED_RUN_SHA, \'Generation configuration hash differs.\')\n    data = {}\n    common_ids = None\n    for src, tgt in DIRECTIONS:\n        direction = f\'{src}->{tgt}\'\n        pair = {}\n        for label in [\'unsplit_control\', \'sentence_split\']:\n            name = f\'nllb3.3b_{src}_{tgt}_dev_{label}.jsonl\'\n            path = input_dir / name\n            require(path.is_file(), f\'Missing saved baseline: {name}\')\n            actual = hashlib.sha256(path.read_bytes()).hexdigest()\n            require(actual == EXPECTED_HASHES[name], f\'Checksum mismatch: {name}\')\n            require(summary[\'directions\'][direction][\'file_sha256\'][name] == actual,\n                    f\'Summary checksum differs: {name}\')\n            with path.open(encoding=\'utf-8\') as f:\n                rows = sorted([json.loads(line) for line in f if line.strip()],\n                              key=lambda r: r[\'sentence_id\'])\n            ids = [r[\'sentence_id\'] for r in rows]\n            require(len(rows) == len(set(ids)) == 997, f\'Bad row IDs: {name}\')\n            if common_ids is None:\n                common_ids = ids\n            require(ids == common_ids, f\'ID alignment differs: {name}\')\n            for row in rows:\n                require(row[\'split\'] == \'dev\', \'Only dev is permitted.\')\n                require(row[\'direction\'] == direction, \'Direction field differs.\')\n                require(row[\'run_spec_sha256\'] == EXPECTED_RUN_SHA, \'Row provenance differs.\')\n                require(all(isinstance(row[k], str) and row[k].strip() for k in\n                            [\'source_text\', \'reference_text\', \'nllb_translation\']),\n                        f\'Empty or invalid text: {name}\')\n            pair[label] = rows\n        u, s = pair[\'unsplit_control\'], pair[\'sentence_split\']\n        changed, multi = [], []\n        for a, b in zip(u, s):\n            require(all(a[k] == b[k] for k in\n                        [\'sentence_id\', \'source_text\', \'reference_text\']),\n                    \'Paired sources/references differ.\')\n            require(\'\'.join(b[\'source_segments\']) == b[\'source_text\'],\n                    \'Source reconstruction differs.\')\n            changed.append(a[\'nllb_translation\'] != b[\'nllb_translation\'])\n            multi.append(b[\'input_segment_count\'] > 1)\n        require(all(not c or m for c, m in zip(changed, multi)),\n                \'A single-sentence item differs across conditions.\')\n        require(sum(changed) == summary[\'directions\'][direction][\n                    \'split_outputs_different_from_current_control\'], \'Changed count differs.\')\n        data[direction] = dict(unsplit=u, split=s, changed=changed, multi=multi)\n    return data, summary\n\n\ndef validate_score_cache(cache, evaluation_sha, item):\n    ids = [r[\'sentence_id\'] for r in item[\'unsplit\']]\n    require(cache[\'evaluation_sha256\'] == evaluation_sha, \'Score provenance differs.\')\n    require(cache[\'sentence_ids\'] == ids, \'Score IDs differ.\')\n    require(cache[\'content_sha256\'] == digest({k: v for k, v in cache.items()\n                                             if k != \'content_sha256\'}),\n            \'Score cache checksum differs.\')\n    u, s = cache[\'unsplit_scores_native\'], cache[\'split_scores_native\']\n    require(len(u) == len(s) == 997, \'Score counts differ.\')\n    require(all(isinstance(x, (int, float)) and math.isfinite(x) for x in u + s),\n            \'Invalid COMET scores.\')\n    require(all(c or a == b for c, a, b in zip(item[\'changed\'], u, s)),\n            \'Identical translations have different cached scores.\')\n    return u, s\n\n\ndef find_completed_evaluation(input_dir, data):\n    """Reuse a validated completed evaluation, including yesterday\'s script."""\n    candidates = sorted((Path(input_dir) / \'evaluation\').glob(\'*/dev_baseline_metrics.json\'),\n                        key=lambda p: p.stat().st_mtime, reverse=True)\n    for path in candidates:\n        try:\n            report = read_json(path)\n            spec = read_json(path.parent / \'evaluation_spec.json\')\n            sha = digest(spec)\n            require(report[\'status\'] == \'dev_baseline_evaluation_complete\', \'Incomplete report.\')\n            require(report[\'evaluation_sha256\'] == sha, \'Evaluation signature differs.\')\n            require(spec[\'generation_run_sha256\'] == EXPECTED_RUN_SHA, \'Wrong generation run.\')\n            require(spec[\'input_file_hashes\'] == EXPECTED_HASHES, \'Wrong input hashes.\')\n            require(spec[\'comet_model\'] == COMET_ID and spec[\'comet_revision\'] == COMET_REV,\n                    \'Wrong COMET checkpoint.\')\n            require(spec[\'bootstrap_resamples\'] == 10000, \'Unexpected bootstrap count.\')\n            require(\'word_order=2\' in spec[\'chrf\'], \'Wrong chrF word order.\')\n            for src, tgt in DIRECTIONS:\n                direction = f\'{src}->{tgt}\'\n                result = report[\'results\'][direction]\n                require(result[\'n\'] == 997, \'Wrong reported count.\')\n                require(result[\'text_outputs_changed\'] == sum(data[direction][\'changed\']),\n                        \'Reported changed count differs.\')\n                require(result[\'multi_sentence_items\'] == sum(data[direction][\'multi\']),\n                        \'Reported multi-sentence count differs.\')\n                cache = read_json(path.parent / f\'scores_{src}_{tgt}_dev.json\')\n                u, s = validate_score_cache(cache, sha, data[direction])\n                for label, scores in [(\'current_unsplit\', u), (\'sentence_split\', s)]:\n                    metrics = result[label]\n                    require(abs(metrics[\'COMET_points\'] - sum(scores) / 997 * 100) < 1e-8,\n                            \'Reported mean COMET differs from saved scores.\')\n                    require(all(math.isfinite(metrics[k]) for k in [\'BLEU\', \'chrF++\']),\n                            \'Invalid lexical metric.\')\n                    require(\'nw:2\' in metrics[\'chrF_signature\'] and\n                            \'nc:6\' in metrics[\'chrF_signature\'], \'Unexpected chrF signature.\')\n                    require(\'tok:13a\' in metrics[\'BLEU_signature\'], \'Unexpected BLEU tokenizer.\')\n                difference = result[\'paired_delta_all_items\']\n                expected_mean = sum(b - a for a, b in zip(u, s)) / 997 * 100\n                require(difference[\'n\'] == 997 and\n                        abs(difference[\'mean_difference_points\'] - expected_mean) < 1e-8,\n                        \'Reported paired mean differs from saved scores.\')\n                lo, hi = difference[\'ci95_points\']\n                require(math.isfinite(lo) and math.isfinite(hi) and lo <= hi,\n                        \'Invalid reported confidence interval.\')\n            return path, report\n        except (OSError, ValueError, KeyError, TypeError) as exc:\n            print(f\'Existing evaluation not reused: {path.parent.name}: {exc}\', flush=True)\n    return None, None\n\n\ndef paired_mean_ci(differences, resamples=10000, seed=42):\n    import numpy as np\n    values = np.asarray(differences, dtype=np.float64)\n    require(values.ndim == 1 and len(values) > 0 and np.isfinite(values).all(),\n            \'Invalid bootstrap input.\')\n    rng = np.random.default_rng(seed)\n    means = []\n    for start in range(0, resamples, 128):\n        count = min(128, resamples - start)\n        indices = rng.integers(0, len(values), size=(count, len(values)))\n        means.append(values[indices].mean(axis=1))\n    low, high = np.quantile(np.concatenate(means), [0.025, 0.975])\n    return dict(n=len(values), mean_difference_points=float(values.mean() * 100),\n                ci95_points=[float(low * 100), float(high * 100)])\n\n\ndef main():\n    import numpy as np\n    import torch\n    from comet import load_from_checkpoint\n    from huggingface_hub import snapshot_download\n    from sacrebleu.metrics import BLEU, CHRF\n\n    root = Path(\'/content/drive/MyDrive/norris\')\n    input_dir = root / \'outputs/generations/baselines\' / RUN_NAME\n    data, generation_summary = load_inputs(input_dir)\n    completed_path, completed_report = find_completed_evaluation(input_dir, data)\n    if completed_path:\n        print(\'DEV BASELINE EVALUATION ALREADY COMPLETE\')\n        print(json.dumps(completed_report[\'results\'], indent=2))\n        print(\'RESULT DIRECTORY:\', completed_path.parent)\n        return\n\n    require(sys.version_info[:2] == (3, 11), \'Use the isolated Python 3.11 environment.\')\n    require(torch.cuda.is_available(), \'Connect an A100 GPU runtime before scoring.\')\n    torch.manual_seed(42)\n    torch.cuda.manual_seed_all(42)\n    torch.set_float32_matmul_precision(\'highest\')\n    environment = {\'python\': sys.version.split()[0], **{name: version(name) for name in\n        [\'torch\', \'transformers\', \'unbabel-comet\', \'numpy\', \'sacrebleu\', \'huggingface-hub\',\n         \'pytorch-lightning\', \'torchmetrics\', \'sentencepiece\']},\n        \'gpu\': torch.cuda.get_device_name(0)}\n    freeze = sorted({f"{d.metadata[\'Name\']}=={d.version}" for d in distributions()\n                     if d.metadata.get(\'Name\')})\n    spec = dict(generation_run_sha256=EXPECTED_RUN_SHA, input_file_hashes=EXPECTED_HASHES,\n        comet_model=COMET_ID, comet_revision=COMET_REV, comet_batch_size=64,\n        length_batching=True, mc_dropout=0, num_workers=2,\n        float32_matmul_precision=\'highest\', bootstrap_resamples=10000, bootstrap_seed=42,\n        bootstrap_method=\'paired percentile CI for mean COMET difference\',\n        difference_orientation=\'sentence_split minus current_unsplit\',\n        comet_report_scale=\'native score multiplied by 100\',\n        bleu=\'13a; mixed case; exponential smoothing\',\n        chrf=\'char_order=6; word_order=2; beta=2\', environment=environment,\n        resolved_packages_sha256=digest(freeze),\n        evaluator_script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())\n    sha = digest(spec)\n    output_dir = input_dir / \'evaluation\' / sha[:16]\n    output_dir.mkdir(parents=True, exist_ok=True)\n    atomic_json(output_dir / \'evaluation_spec.json\', spec)\n    (output_dir / \'requirements_resolved.txt\').write_text(\'\\n\'.join(freeze) + \'\\n\')\n    model = None\n\n    def predict(rows):\n        nonlocal model\n        if not rows:\n            return np.empty(0, dtype=np.float64)\n        if model is None:\n            snapshot = Path(snapshot_download(repo_id=COMET_ID, revision=COMET_REV,\n                allow_patterns=[\'checkpoints/model.ckpt\', \'hparams.yaml\']))\n            model = load_from_checkpoint(str(snapshot / \'checkpoints/model.ckpt\'))\n            model.eval()\n            encoder = getattr(getattr(model, \'encoder\', None), \'model\', None)\n            atomic_json(output_dir / \'model_provenance.json\', dict(\n                comet_model=COMET_ID, comet_revision=COMET_REV,\n                encoder_config_commit=getattr(getattr(encoder, \'config\', None), \'_commit_hash\', None)))\n        prediction = model.predict([dict(src=r[\'source_text\'], mt=r[\'nllb_translation\'],\n            ref=r[\'reference_text\']) for r in rows], batch_size=64, gpus=1,\n            progress_bar=False, num_workers=2, length_batching=True, mc_dropout=0)\n        scores = np.asarray(prediction.scores, dtype=np.float64)\n        require(len(scores) == len(rows) and np.isfinite(scores).all(), \'Invalid COMET output.\')\n        return scores\n\n    def metrics(rows, scores):\n        hypotheses = [r[\'nllb_translation\'] for r in rows]\n        references = [[r[\'reference_text\'] for r in rows]]\n        bleu, chrf = BLEU(tokenize=\'13a\'), CHRF(char_order=6, word_order=2, beta=2)\n        b, c = bleu.corpus_score(hypotheses, references), chrf.corpus_score(hypotheses, references)\n        return {\'BLEU\': float(b.score), \'chrF++\': float(c.score),\n                \'COMET_points\': float(scores.mean() * 100),\n                \'BLEU_signature\': str(bleu.get_signature()), \'chrF_signature\': str(chrf.get_signature())}\n\n    results = {}\n    for src, tgt in DIRECTIONS:\n        direction = f\'{src}->{tgt}\'\n        item = data[direction]\n        u, s = item[\'unsplit\'], item[\'split\']\n        changed = np.asarray(item[\'changed\'], dtype=bool)\n        multi = np.asarray(item[\'multi\'], dtype=bool)\n        path = output_dir / f\'scores_{src}_{tgt}_dev.json\'\n        if path.exists():\n            a, b = validate_score_cache(read_json(path), sha, item)\n            a, b = np.asarray(a), np.asarray(b)\n            print(\'RESUMED:\', direction, flush=True)\n        else:\n            print(\'SCORING:\', direction, flush=True)\n            a = predict(u)\n            b = a.copy()\n            indices = np.flatnonzero(changed)\n            b[indices] = predict([s[int(i)] for i in indices])\n            cache = dict(evaluation_sha256=sha, sentence_ids=[r[\'sentence_id\'] for r in u],\n                         unsplit_scores_native=a.tolist(), split_scores_native=b.tolist())\n            cache[\'content_sha256\'] = digest(cache)\n            validate_score_cache(cache, sha, item)\n            atomic_json(path, cache)\n        delta = b - a\n        result = dict(n=997, multi_sentence_items=int(multi.sum()),\n            text_outputs_changed=int(changed.sum()),\n            current_unsplit=metrics(u, a), sentence_split=metrics(s, b),\n            paired_delta_all_items=paired_mean_ci(delta),\n            paired_delta_multi_sentence_items=paired_mean_ci(delta[multi]),\n            metric_change_bins_native_threshold_002={\n                \'increased_gt_0.02\': int((delta > .02).sum()),\n                \'within_inclusive_plus_minus_0.02\': int((np.abs(delta) <= .02).sum()),\n                \'decreased_gt_0.02\': int((delta < -.02).sum())},\n            sentence_split_baseline_pools={\'COMET_lt_0.80\': int((b < .8).sum()),\n                                          \'COMET_ge_0.90\': int((b >= .9).sum())})\n        old_path = root / \'outputs/logs\' / f\'comet_seg_nllb_{src}_{tgt}_dev.json\'\n        if old_path.exists():\n            old = read_json(old_path)\n            old_scores = np.asarray([old[str(r[\'sentence_id\'])] for r in u], dtype=np.float64)\n            require(np.isfinite(old_scores).all(), \'Invalid saved Phase 2 COMET scores.\')\n            diff = a - old_scores\n            result[\'phase2_comet_recheck\'] = dict(mean_difference_points=float(diff.mean() * 100),\n                max_absolute_difference_native=float(np.abs(diff).max()),\n                mean_absolute_difference_native=float(np.abs(diff).mean()))\n        results[direction] = result\n        print(\'DONE:\', direction, \'| COMET unsplit / split:\',\n              round(result[\'current_unsplit\'][\'COMET_points\'], 4),\n              round(result[\'sentence_split\'][\'COMET_points\'], 4), flush=True)\n    report = dict(status=\'dev_baseline_evaluation_complete\',\n        interpretation=\'Exploratory dev baseline diagnostic; not the LLM pilot or devtest evaluation.\',\n        evaluation_sha256=sha, environment=environment, bootstrap_resamples=10000,\n        difference_orientation=\'sentence_split minus current_unsplit\', results=results)\n    atomic_json(output_dir / \'dev_baseline_metrics.json\', report)\n    print(\'\\nDEV BASELINE EVALUATION COMPLETE\')\n    print(json.dumps(results, indent=2))\n    print(\'\\nRESULT DIRECTORY:\', output_dir)\n    print(\'Devtest was not opened.\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
source_sha = hashlib.sha256(EVALUATOR_SOURCE.encode('utf-8')).hexdigest()
SCRIPT_PATH = ROOT / 'code/src/evaluation' / f'evaluate_dev_baselines_resume_{source_sha[:16]}.py'
SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
if SCRIPT_PATH.exists():
    if SCRIPT_PATH.read_text(encoding='utf-8') != EVALUATOR_SOURCE:
        raise RuntimeError('Saved evaluator source differs; stopped before overwriting it.')
else:
    temporary = SCRIPT_PATH.with_suffix('.py.tmp')
    temporary.write_text(EVALUATOR_SOURCE, encoding='utf-8')
    os.replace(temporary, SCRIPT_PATH)

module_spec = importlib.util.spec_from_file_location('norris_resume_evaluator', SCRIPT_PATH)
probe = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(probe)
DEV_DATA, GENERATION_SUMMARY = probe.load_inputs(RUN_DIR)
COMPLETED_PATH, COMPLETED_REPORT = probe.find_completed_evaluation(RUN_DIR, DEV_DATA)

print('PASS: all eight saved dev files match their recorded SHA-256 hashes')
print('PASS: 997 aligned items per direction per condition')
for direction, record in GENERATION_SUMMARY['directions'].items():
    print(direction, '| archived exact matches:', record['archived_unsplit_exact_matches'],
          '| split text changes:', record['split_outputs_different_from_current_control'])

if COMPLETED_PATH:
    print('BASELINE EVALUATION ALREADY COMPLETE:', COMPLETED_PATH)
    print('The next two cells will skip installation and scoring.')
else:
    print('BASELINE EVALUATION PENDING — continue to cell 2.')

print('Devtest was not opened. No NLLB generation was performed.')


In [ ]:
# 2 — Build/verify an isolated evaluation environment only if needed.
import gc
import shutil
import subprocess
import sys
import os

EVAL_PY = Path('/content/norris_evalenv/bin/python')

if COMPLETED_REPORT is not None:
    print('SKIPPED: a completed baseline evaluation is already verified.')
else:
    if 'nllb' in globals():
        del nllb
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except ImportError:
        pass

    install_env = os.environ.copy()
    for key in ['PYTHONPATH', 'UV_SYSTEM_PYTHON', 'VIRTUAL_ENV']:
        install_env.pop(key, None)

    def run_setup(arguments):
        subprocess.run(arguments, check=True, env=install_env)

    run_setup([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    if not EVAL_PY.exists():
        run_setup([sys.executable, '-m', 'uv', 'venv', '--python', '3.11',
                   '/content/norris_evalenv'])
    run_setup([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(EVAL_PY),
               'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
    run_setup([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(EVAL_PY),
               'torch==2.6.0', 'unbabel-comet==2.2.7', 'transformers==4.49.0',
               'huggingface-hub==0.36.0', 'numpy==1.26.4', 'sacrebleu==2.6.0',
               'torchmetrics==0.10.3', 'setuptools==80.9.0'])
    run_setup([str(EVAL_PY), '-c',
               "import sys, torch, comet, transformers, numpy; "
               "print('PYTHON:', sys.version.split()[0]); "
               "print('TORCH:', torch.__version__); "
               "print('TRANSFORMERS:', transformers.__version__); "
               "print('NUMPY:', numpy.__version__); "
               "print('CUDA:', torch.cuda.is_available()); "
               "assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'"])
    print('EVALUATION ENVIRONMENT READY')


In [ ]:
# 3 — Run pending scoring; direction-level score checkpoints are saved.
from datetime import datetime, timezone
import subprocess
import os

if COMPLETED_REPORT is not None:
    print('SKIPPED: completed evaluation reused.')
else:
    log_dir = ROOT / 'outputs/logs/phase3'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / ('evaluation_resume_' +
                         datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.log')
    process_env = os.environ.copy()
    process_env.pop('PYTHONPATH', None)
    process_env['TOKENIZERS_PARALLELISM'] = 'false'
    print('Running evaluator:', SCRIPT_PATH.name)
    print('Log:', log_path)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen([str(EVAL_PY), '-u', str(SCRIPT_PATH)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            bufsize=1, env=process_env)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Evaluation stopped (exit {return_code}). '
                           f'Send the error from this cell. Log: {log_path}')
    COMPLETED_PATH, COMPLETED_REPORT = probe.find_completed_evaluation(RUN_DIR, DEV_DATA)
    if COMPLETED_PATH is None:
        raise RuntimeError('No validated completed evaluation was found after scoring.')

print('DEV BASELINE EVALUATION VERIFIED')
print(json.dumps(COMPLETED_REPORT['results'], indent=2))
print('RESULT DIRECTORY:', COMPLETED_PATH.parent)


In [ ]:
# 4 — Package the evidence for review. This is not the final Phase 3 report.
import zipfile
import hashlib
from datetime import datetime, timezone

if COMPLETED_PATH is None or COMPLETED_REPORT is None:
    raise RuntimeError('Complete the preceding cells first.')

evaluation_dir = COMPLETED_PATH.parent
evaluation_spec = probe.read_json(evaluation_dir / 'evaluation_spec.json')
expected_source_sha = evaluation_spec.get('evaluator_script_sha256')
generating_source = None
for candidate in [SCRIPT_PATH, ROOT / 'code/src/evaluation/evaluate_dev_baselines.py']:
    if candidate.is_file() and hashlib.sha256(candidate.read_bytes()).hexdigest() == expected_source_sha:
        generating_source = candidate
        break

evidence_dir = ROOT / 'outputs/reports/phase3_evidence'
evidence_dir.mkdir(parents=True, exist_ok=True)
evaluation_sha = COMPLETED_REPORT['evaluation_sha256']
bundle = evidence_dir / f'Norris_Phase3_Dev_Evaluation_Evidence_{evaluation_sha[:16]}.zip'
status = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'generation_run_sha256': probe.EXPECTED_RUN_SHA,
    'evaluation_sha256': evaluation_sha,
    'baseline_generation': 'verified',
    'baseline_evaluation': 'verified_complete',
    'generating_evaluator_source_verified': generating_source is not None,
    'post_editing_pilots': 'pending_evidence',
    'ICAI_overlap_audit': 'pending_final_ICAI_paper',
    'protocol_freeze': 'not_declared',
    'phase3_report': 'not_finalized',
    'scope': 'Dev baseline evaluation only; no devtest scoring.',
}

with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(RUN_DIR / 'generation_summary.json', 'generation/generation_summary.json')
    archive.write(RUN_DIR / 'run_spec.json', 'generation/run_spec.json')
    for name in ['dev_baseline_metrics.json', 'evaluation_spec.json',
                 'requirements_resolved.txt', 'model_provenance.json']:
        path = evaluation_dir / name
        if path.is_file():
            archive.write(path, 'evaluation/' + name)
    for src, tgt in probe.DIRECTIONS:
        name = f'scores_{src}_{tgt}_dev.json'
        archive.write(evaluation_dir / name, 'evaluation/' + name)
    archive.write(SCRIPT_PATH, 'code/resume_evaluator_source.py')
    if generating_source is not None:
        archive.write(generating_source, 'code/generating_evaluator_source.py')
    archive.writestr('stage_status.json', json.dumps(status, indent=2))

with zipfile.ZipFile(bundle) as archive:
    if archive.testzip() is not None:
        raise RuntimeError('Evidence ZIP integrity check failed.')

print('BASELINE EVALUATION EVIDENCE READY')
print('Attach this ZIP in the chat:', bundle.name)
print('Saved at:', bundle)
print('Next: audit metrics, implement retrieval/prompts, and run the dev LLM pilots.')
from google.colab import files
files.download(str(bundle))


## Continue the research after the evidence ZIP is reviewed

- Confirm the split baseline scores, paired confidence intervals, and Phase 2 COMET consistency.
- Implement the three post-editing prompts and schema constraints; construct dev-only LaBSE/FAISS retrieval with pilot-ID exclusion.
- Record a fixed, balanced dev pilot sample and run the model/strategy matrix with raw responses, retries, timing, token usage, and failure flags.
- Audit pilot metrics, preservation flags, and harmful edits. Brief diagnostics are recorded separately from edited translations.
- Finish the final ICAI overlap comparison. Exclude the J.UCS manuscript, per the PI's instructions.
- Resolve the GPT API component, record all deviations, then freeze the protocol before devtest scoring.
- Generate Report 3 from the verified artifacts; use the existing ICNLP Overleaf project for later manuscript integration.

The full two-model pilot target is 100 dev IDs × 4 directions × 3 strategies × 2 models = 2,400 outputs, excluding smoke tests and retries. A baseline generation run is not this pilot.
